# Etapa 2: CNN Profunda desde Cero

**Objetivo**: Diseñar, entrenar y evaluar una red neuronal convolucional (CNN) profunda construida desde cero para la clasificación de defectos en piel de aeronaves.

**Contenido**:
1. Setup e importaciones
2. Carga de datos con DataLoaders (imágenes RGB, 224×224)
3. CNN de 4 bloques (arquitectura base)
4. CNN de 6 bloques (más profunda) — Ablation Study
5. Ablation Study: efecto de capas, dropout, learning rate
6. Evaluación comparativa CNN vs MLP
7. Visualización con Grad-CAM
8. Conclusiones

## 1. Setup e Importaciones

In [ ]:
import sys
sys.path.insert(0, "..")

import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.data_utils import (
    get_dataloaders, get_transforms, CLASS_NAMES,
    get_class_weights, IMAGENET_MEAN, IMAGENET_STD,
)
from src.models import CNNClassifier, CNNClassifierDeep, count_parameters, get_model
from src.train import train_classifier
from src.evaluate import (
    compute_metrics, get_predictions_with_proba,
    plot_confusion_matrix, plot_training_history,
    plot_roc_curves, plot_model_comparison, print_comparison_table,
)
from src.gradcam import generate_gradcam, plot_gradcam, get_target_layer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 42
FIGURES_DIR = Path("../figures/etapa2")
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR = Path("../models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"Device: {DEVICE}")

## 2. Carga de Datos

In [ ]:
train_transform = get_transforms(img_size=224, augment=True)
val_transform = get_transforms(img_size=224, augment=False)

data_dir = Path("../data/splits")
class_weights = get_class_weights(data_dir / "train")

train_loader, val_loader, test_loader = get_dataloaders(
    data_dir=data_dir,
    train_transform=train_transform,
    val_transform=val_transform,
    batch_size=32,
    num_workers=2,
)

print(f"Train: {len(train_loader.dataset)} | Val: {len(val_loader.dataset)} | Test: {len(test_loader.dataset)}")
print(f"Class weights: {class_weights}")

## 3. CNN Base (4 bloques convolucionales)

Arquitectura: 4 × (Conv3×3 → BatchNorm → ReLU → MaxPool2×2) + GlobalAvgPool + FC(256 → 5)
- Preserva estructura espacial (a diferencia del MLP)
- Parámetros compartidos vía kernels convolucionales
- Pooling reduce dimensiones gradualmente

In [ ]:
cnn_model = CNNClassifier(num_classes=5, dropout=0.5)
cnn_params = count_parameters(cnn_model)
print(f"CNN Base (4 bloques):")
print(f"  Parametros totales:     {cnn_params['total']:,}")
print(f"  Parametros entrenables: {cnn_params['trainable']:,}")
print(f"\nArquitectura:\n{cnn_model}")

In [ ]:
history_cnn = train_classifier(
    model=cnn_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=40,
    lr=1e-3,
    weight_decay=1e-4,
    class_weights=class_weights,
    device=DEVICE,
    save_dir=MODELS_DIR,
    model_name="cnn_base",
    patience=10,
    scheduler_type="cosine",
)

In [ ]:
plot_training_history(history_cnn, title="CNN Base (4 bloques)", save_path=FIGURES_DIR / "02_cnn_base_history.png")

## 4. CNN Profunda (6 bloques) — Más capacidad

Se añaden 2 bloques convolucionales adicionales (256→512→512) para evaluar si más profundidad mejora la captura de patrones de defectos.

In [ ]:
cnn_deep_model = CNNClassifierDeep(num_classes=5, dropout=0.5)
cnn_deep_params = count_parameters(cnn_deep_model)
print(f"CNN Profunda (6 bloques):")
print(f"  Parametros totales:     {cnn_deep_params['total']:,}")
print(f"  Parametros entrenables: {cnn_deep_params['trainable']:,}")
print(f"\nComparacion:")
print(f"  CNN Base:     {cnn_params['total']:>10,} params")
print(f"  CNN Profunda: {cnn_deep_params['total']:>10,} params")
print(f"  Ratio:        {cnn_deep_params['total']/cnn_params['total']:.1f}x")

In [ ]:
history_cnn_deep = train_classifier(
    model=cnn_deep_model,
    train_loader=train_loader,
    val_loader=val_loader,
    num_epochs=40,
    lr=1e-3,
    weight_decay=1e-4,
    class_weights=class_weights,
    device=DEVICE,
    save_dir=MODELS_DIR,
    model_name="cnn_deep",
    patience=10,
    scheduler_type="cosine",
)

In [ ]:
plot_training_history(history_cnn_deep, title="CNN Profunda (6 bloques)", save_path=FIGURES_DIR / "02_cnn_deep_history.png")

## 5. Ablation Study

Evaluamos el impacto de diferentes hiperparámetros en la CNN base:
1. **Número de capas**: 4 bloques vs 6 bloques (ya entrenados arriba)
2. **Dropout**: 0.3 vs 0.5 vs 0.7
3. **Learning rate**: 1e-2 vs 1e-3 vs 1e-4

In [ ]:
ablation_results = {}
ablation_results["4-blocks_d0.5_lr1e-3"] = history_cnn
ablation_results["6-blocks_d0.5_lr1e-3"] = history_cnn_deep

for dropout_val in [0.3, 0.7]:
    print(f"\n{'='*60}")
    print(f"Ablation: CNN Base con dropout={dropout_val}")
    print(f"{'='*60}")
    model_abl = CNNClassifier(num_classes=5, dropout=dropout_val)
    hist = train_classifier(
        model=model_abl,
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=25,
        lr=1e-3,
        class_weights=class_weights,
        device=DEVICE,
        save_dir=MODELS_DIR,
        model_name=f"cnn_abl_d{dropout_val}",
        patience=7,
        scheduler_type="cosine",
    )
    ablation_results[f"4-blocks_d{dropout_val}_lr1e-3"] = hist

for lr_val in [1e-2, 1e-4]:
    print(f"\n{'='*60}")
    print(f"Ablation: CNN Base con lr={lr_val}")
    print(f"{'='*60}")
    model_abl = CNNClassifier(num_classes=5, dropout=0.5)
    hist = train_classifier(
        model=model_abl,
        train_loader=train_loader,
        val_loader=val_loader,
        num_epochs=25,
        lr=lr_val,
        class_weights=class_weights,
        device=DEVICE,
        save_dir=MODELS_DIR,
        model_name=f"cnn_abl_lr{lr_val}",
        patience=7,
        scheduler_type="cosine",
    )
    ablation_results[f"4-blocks_d0.5_lr{lr_val}"] = hist

print("\nAblation study completado")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

configs = list(ablation_results.keys())
best_accs = [ablation_results[c]["best_val_acc"] for c in configs]

ax = axes[0]
bars = ax.barh(configs, best_accs, color=plt.cm.viridis(np.linspace(0.2, 0.8, len(configs))))
ax.set_xlabel("Best Val Accuracy")
ax.set_title("Ablation Study: Mejor Accuracy de Validacion")
for bar, acc in zip(bars, best_accs):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height()/2,
            f"{acc:.3f}", va='center', fontsize=10)
ax.set_xlim(0, max(best_accs) * 1.15)

ax = axes[1]
for name, hist in ablation_results.items():
    ax.plot(hist["val_acc"], label=name, alpha=0.8)
ax.set_xlabel("Epoch")
ax.set_ylabel("Val Accuracy")
ax.set_title("Ablation Study: Curvas de Validacion")
ax.legend(fontsize=8, loc="lower right")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_ablation_study.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Evaluación en Test Set

### 6.1 Mejor CNN (seleccionada del ablation study)

In [ ]:
best_config = max(ablation_results, key=lambda k: ablation_results[k]["best_val_acc"])
print(f"Mejor configuracion: {best_config}")
print(f"  Best Val Acc: {ablation_results[best_config]['best_val_acc']:.4f}")

if "6-blocks" in best_config:
    best_cnn = CNNClassifierDeep(num_classes=5, dropout=0.5)
    best_cnn.load_state_dict(torch.load(MODELS_DIR / "cnn_deep_best.pt", weights_only=True))
else:
    best_cnn = CNNClassifier(num_classes=5, dropout=0.5)
    best_cnn.load_state_dict(torch.load(MODELS_DIR / "cnn_base_best.pt", weights_only=True))

best_cnn = best_cnn.to(DEVICE)
best_cnn.eval()

y_true, y_pred, y_proba = get_predictions_with_proba(best_cnn, test_loader, DEVICE)
metrics_cnn = compute_metrics(y_true, y_pred, y_proba, CLASS_NAMES)
print(f"\nMetricas en Test Set:")
for k, v in metrics_cnn.items():
    if isinstance(v, float):
        print(f"  {k}: {v:.4f}")

### 6.2 Matriz de Confusión y Curvas ROC

In [ ]:
plot_confusion_matrix(y_true, y_pred, CLASS_NAMES, title="Mejor CNN", save_path=FIGURES_DIR / "02_cnn_confusion_matrix.png")
plot_roc_curves(y_true, y_proba, CLASS_NAMES, title="Mejor CNN", save_path=FIGURES_DIR / "02_cnn_roc_curves.png")

### 6.3 Comparación CNN vs MLP Base

In [ ]:
import json

mlp_metrics_path = MODELS_DIR / "mlp_base_metrics.json"
if mlp_metrics_path.exists():
    with open(mlp_metrics_path) as f:
        metrics_mlp = json.load(f)
else:
    print("No se encontraron metricas del MLP. Ejecuta primero Notebook 01.")
    metrics_mlp = {"accuracy": 0.0, "f1_weighted": 0.0, "roc_auc_ovr": 0.0}

comparison = {
    "MLP Base": metrics_mlp,
    "Mejor CNN": metrics_cnn,
}
print_comparison_table(comparison)

model_names = list(comparison.keys())
metric_names = ["accuracy", "f1_weighted", "roc_auc_ovr"]
plot_model_comparison(comparison, metric_names, save_path=FIGURES_DIR / "02_cnn_vs_mlp.png")

## 7. Visualización con Grad-CAM

Grad-CAM nos permite ver qué regiones de la imagen son más importantes para la decisión del modelo CNN.

In [ ]:
from torchvision import transforms

test_dataset = test_loader.dataset
samples_per_class = {}
for idx in range(len(test_dataset)):
    img, label = test_dataset[idx]
    if label not in samples_per_class:
        samples_per_class[label] = (img, label)
    if len(samples_per_class) == len(CLASS_NAMES):
        break

fig, axes = plt.subplots(len(samples_per_class), 3, figsize=(12, 4 * len(samples_per_class)))
target_layer = get_target_layer(best_cnn, model_type="cnn")

for i, (label_id, (img_tensor, label)) in enumerate(sorted(samples_per_class.items())):
    img_show = img_tensor.clone()
    for c in range(3):
        img_show[c] = img_show[c] * IMAGENET_STD[c] + IMAGENET_MEAN[c]
    img_show = img_show.permute(1, 2, 0).numpy().clip(0, 1)

    cam_image = generate_gradcam(best_cnn, img_tensor.unsqueeze(0), target_layer, device=DEVICE)

    with torch.no_grad():
        logits = best_cnn(img_tensor.unsqueeze(0).to(DEVICE))
        pred = logits.argmax(1).item()

    axes[i, 0].imshow(img_show)
    axes[i, 0].set_title(f"Original: {CLASS_NAMES[label]}")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(cam_image)
    axes[i, 1].set_title(f"Grad-CAM -> Pred: {CLASS_NAMES[pred]}")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(img_show)
    axes[i, 2].imshow(cam_image, alpha=0.5)
    axes[i, 2].set_title("Overlay")
    axes[i, 2].axis("off")

plt.suptitle("Grad-CAM: Regiones de atencion de la CNN", fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "02_gradcam_cnn.png", dpi=150, bbox_inches="tight")
plt.show()

## 8. Guardar Resultados

In [ ]:
cnn_metrics_path = MODELS_DIR / "cnn_best_metrics.json"
metrics_json = {k: float(v) if hasattr(v, 'item') else v for k, v in metrics_cnn.items()}
with open(cnn_metrics_path, 'w') as f:
    json.dump(metrics_json, f, indent=2)

ablation_summary = {name: {"best_val_acc": h["best_val_acc"]} for name, h in ablation_results.items()}
with open(MODELS_DIR / "ablation_results.json", 'w') as f:
    json.dump(ablation_summary, f, indent=2)

print(f"Metricas CNN guardadas en {cnn_metrics_path}")
print(f"Ablation results guardados")

## 9. Conclusiones

### Resultados del Ablation Study
- Se evaluaron variaciones en profundidad (4 vs 6 bloques), dropout (0.3, 0.5, 0.7) y learning rate (1e-2, 1e-3, 1e-4)
- La CNN aprovecha la **estructura espacial** de las imágenes, mejorando significativamente sobre el MLP

### CNN vs MLP
- La CNN preserva las relaciones espaciales entre píxeles vecinos, crucial para detectar patrones de defectos
- Los kernels convolucionales actúan como detectores de bordes, texturas y patrones locales
- Grad-CAM confirma que la CNN atiende a las regiones relevantes del defecto

### Próximos pasos
- **Etapa 3**: Transfer learning con ResNet50 y ViT, que traen conocimiento preentrenado de ImageNet
- Se espera mejora adicional al aprovechar features aprendidas en millones de imágenes